# ko2en
baseline에서 바꾼 것
- 설정: SRC, TGT = 'ko', 'en' / hid_dim 128 / dropout 0.3 / EPOCHS 40
- 입력 정규화 함수 normalize(text, lang) 추가
- en2ko 증강 데이터를 뒤집어서 extra_train으로 넣고 다시 학습, test가 높은 쪽 저장

원본 en-ko 데이터는 대부분 4자리 숫자라 긴 숫자에 약해서 증강 데이터로 보완

In [1]:
# 설정
SRC, TGT = 'ko', 'en'  # num, en, ko

DATA_DIR = '../data'
SAVE_DIR = '..'  # pt 저장 위치

CONFIG = dict(
    emb_dim=128,
    hid_dim=128,  # 데이터가 적어서 줄임
    n_layers=2,
    dropout=0.3,
    rnn='gru',  # gru / lstm
    attention='luong',  # 'luong' / None
)
BATCH_SIZE = 128
EPOCHS = 40
EPOCHS_AUG = 10  # 증강은 train이 많아서 epoch 적게
SHORT_MAX_DIGITS, SHORT_REPEAT = 3, 20  # 짧은 숫자 반복 (en2ko와 같음)
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
SEED = 42
MAX_TRAIN = None  # 디버깅용 train 샘플 수

In [2]:
# Colab: txt를 업로드했으면 그걸 쓰고, 아니면 레포 clone
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('/content/num2kor.txt') and os.path.exists('/content/num2txt.txt'):
        %cd /content
        DATA_DIR = '.'; SAVE_DIR = '.'
    else:
        if not os.path.exists('/content/2026-nlp'):
            !git clone https://github.com/jungyy808/2026-nlp.git /content/2026-nlp
        %cd /content/2026-nlp/project01/work
        # 증강 데이터 생성용 pt (main에 없으면 각 팀원 브랜치에서 가져옴)
        for name in ('num2en', 'num2ko'):
            if not os.path.exists(f'../{name}.pt'):
                !git fetch -q origin {name} && git show origin/{name}:project01/{name}.pt > ../{name}.pt

Cloning into '/content/2026-nlp'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 75 (delta 11), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 23.54 MiB | 8.67 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/2026-nlp/project01/work


In [3]:
import os, random, time, zlib, math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## 데이터

In [4]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    # en: 단어 단위 / num, ko: 문자 단위 (공백 포함)
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    # CRLF 제거
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    # (src, tgt, key) 반환, key는 split 기준
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:  # en-ko: 공통 key만 사용
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    # key 해시 기준 8:1:1 분할
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):  # sos/eos 제외
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    # extra_train: 증강 등으로 추가할 (src, tgt, key) 리스트. train에만 들어가고 valid/test는 원본만 사용
    # extra_train 토큰도 vocab에 포함 (억, billion이 원본에 없음. extra_train=None이면 베이스라인과 같음)
    vocab_pairs = pairs + list(extra_train or [])
    src_vocab = Vocab(src_lang, [s for s, _, _ in vocab_pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in vocab_pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]  # valid/test 숫자와 겹치는 건 제외
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)

In [5]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음 (Colab이면 clone 셀 확인)'
print('pt 저장 위치:', os.path.abspath(SAVE_DIR))

pairs = get_pairs(SRC, TGT)
train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN)

MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs) + 10  # 데이터 기준 최대 길이
print(f'{SRC} -> {TGT}: 전체 {len(pairs):,}쌍 | train {len(train_loader.dataset):,} / '
      f'valid {len(valid_loader.dataset):,} / test {len(test_loader.dataset):,}')
print(f'vocab: src {len(src_vocab)}, tgt {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')
for s, t, _ in pairs[:3]:
    print(f'  {s!r} -> {t!r}')

작업 위치: /content/2026-nlp/project01/work
pt 저장 위치: /content/2026-nlp/project01
ko -> en: 전체 12,738쌍 | train 10,162 / valid 1,274 / test 1,302
vocab: src 19, tgt 35 | 디코딩 최대 길이 22
  '영' -> 'zero'
  '일' -> 'one'
  '십' -> 'ten'


## 모델 (GRU/LSTM encoder-decoder + attention)

In [6]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden


class Decoder(nn.Module):
    # bahdanau 분기: num2en.pt 로드용 (김태우 구현, en2ko와 같음)
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention):
        super().__init__()
        assert attention in ('luong', 'bahdanau', None), f'지원하지 않는 attention: {attention}'
        assert rnn in ('gru', 'lstm'), f'지원하지 않는 rnn: {rnn}'
        self.attention = attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        rnn_in = emb_dim + hid_dim if attention == 'bahdanau' else emb_dim
        self.rnn = RNN(rnn_in, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)  # Luong (general)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        elif attention == 'bahdanau':
            self.W_h = nn.Linear(hid_dim, hid_dim, bias=False)
            self.W_s = nn.Linear(hid_dim, hid_dim, bias=False)
            self.v_a = nn.Linear(hid_dim, 1, bias=False)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)
        if self.attention == 'bahdanau':
            query = hidden[0][-1] if isinstance(hidden, tuple) else hidden[-1]
            scores = self.v_a(torch.tanh(self.W_h(enc_out) + self.W_s(query).unsqueeze(1))).squeeze(-1)
            scores = scores.masked_fill(~src_mask, -1e9)
            ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
            h, hidden = self.rnn(torch.cat([x, ctx.unsqueeze(1)], dim=2), hidden)
            h = torch.tanh(self.combine(torch.cat([h.squeeze(1), ctx], dim=1)))
        else:
            h, hidden = self.rnn(x, hidden)
            h = h.squeeze(1)
            if self.attention == 'luong':
                scores = torch.bmm(enc_out, self.attn(h).unsqueeze(2)).squeeze(2)
                scores = scores.masked_fill(~src_mask, -1e9)
                ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
                h = torch.tanh(self.combine(torch.cat([h, ctx], dim=1)))
        return self.out(self.drop(h)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong'):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)  # (hidden, enc_out, mask)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        # tgt[:, 1:] 예측
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    # beam search용 배치 재정렬
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)


model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters()))

params: 456995


## 학습, greedy / beam 디코딩

In [7]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    # batch greedy
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)  # 로드한 모델은 저장된 max_len 사용
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def beam_decode(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams = [([], 0.0)]  # (seq, log prob)
    finished = []
    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / len(seq)))  # 길이 정규화
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams or len(finished) >= beam:
            break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / len(seq)) for seq, score in beams]
    best = max(finished, key=lambda f: f[1])[0]
    return tgt_vocab.decode(best)


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    # exact match 기준
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR, save_path=None):
    # best valid acc 모델 복원
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history

In [8]:
history = train(model, train_loader, valid_loader, EPOCHS, LR)

epoch  1 | train loss 2.1009 | valid loss 1.0135 | valid acc 37.21% | 11s
epoch  2 | train loss 0.6097 | valid loss 0.1956 | valid acc 82.81% | 4s
epoch  3 | train loss 0.2215 | valid loss 0.0876 | valid acc 88.78% | 4s
epoch  4 | train loss 0.1256 | valid loss 0.0598 | valid acc 90.74% | 4s
epoch  5 | train loss 0.0934 | valid loss 0.0520 | valid acc 91.37% | 5s
epoch  6 | train loss 0.0829 | valid loss 0.0425 | valid acc 92.23% | 4s
epoch  7 | train loss 0.0625 | valid loss 0.0372 | valid acc 93.25% | 5s
epoch  8 | train loss 0.0557 | valid loss 0.0325 | valid acc 94.74% | 4s
epoch  9 | train loss 0.0492 | valid loss 0.0280 | valid acc 96.47% | 4s
epoch 10 | train loss 0.0456 | valid loss 0.0235 | valid acc 97.10% | 4s
epoch 11 | train loss 0.0313 | valid loss 0.0199 | valid acc 97.88% | 5s
epoch 12 | train loss 0.0318 | valid loss 0.0179 | valid acc 97.72% | 4s
epoch 13 | train loss 0.0316 | valid loss 0.0161 | valid acc 97.88% | 4s
epoch 14 | train loss 0.0238 | valid loss 0.0145 |

## test 평가

In [9]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')


test_rows = predict_all(model, test_loader, tgt_vocab)
print(f'{SRC} -> {TGT} test exact match (greedy): {exact_match(test_rows)*100:.2f}%  (n={len(test_rows):,})\n')
acc_by_length(test_rows)
print()
show_examples(test_rows)

ko -> en test exact match (greedy): 99.54%  (n=1,302)

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |    277 | 100.00%
     6 |     24 |  91.67%
     7 |      4 |  25.00%
     8 |      1 |   0.00%

--- 맞은 예시 ---
입력: 천사
정답: one thousand four
예측: one thousand four

입력: 천십
정답: one thousand ten
예측: one thousand ten

입력: 천십팔
정답: one thousand eighteen
예측: one thousand eighteen

입력: 천십구
정답: one thousand nineteen
예측: one thousand nineteen

입력: 만 이백육
정답: ten thousand two hundred six
예측: ten thousand two hundred six

--- 틀린 예시 ---
입력: 이천구백삼십일만 천삼백십삼
정답: twenty nine million three hundred eleven thousand three hundred thirteen
예측: two thousand nine hundred nineteen

입력: 삼백칠십칠만 사백팔십구
정답: three million seven hundred seventy thousand four hundred eighty nine
예측: three million seventy hundred seventy thousand four hundred eighty nine

입력: 육백육십구만 팔천사백구십육
정답: six million six hundred ninety eight thousand four hundred ninety

In [10]:
# greedy vs beam (test 200개)
sample = test_loader.dataset.pairs[:200]
g = sum(greedy_decode(model, s, src_vocab, tgt_vocab) == t for s, t, _ in sample)
b = sum(beam_decode(model, s, src_vocab, tgt_vocab, BEAM_SIZE) == t for s, t, _ in sample)
print(f'test 200개: greedy {g/len(sample)*100:.1f}% | beam({BEAM_SIZE}) {b/len(sample)*100:.1f}%')

test 200개: greedy 100.0% | beam(5) 100.0%


## 저장

In [11]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    # 저장 형식 고정 (통합 시 로드)
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt


SAVE_PATH = os.path.join(SAVE_DIR, f'{SRC}2{TGT}.pt')
save_model(SAVE_PATH, model, CONFIG, src_vocab, tgt_vocab,
           metrics={'test_exact_match': exact_match(test_rows), 'history': history})

saved: ../ko2en.pt (1.8 MB)


## en2ko 증강 데이터로 다시 학습

원본 모델은 6자리 이상 숫자를 많이 틀림. en2ko(김수현)에서 만든 증강 데이터를 (한글, 영어)로 뒤집어서 extra_train으로 넣음.
- 한쪽 파일에만 있는 train 숫자의 빈 언어를 num2en.pt, num2ko.pt로 생성
- valid/test는 원본 그대로라 같은 test로 비교
- 번역은 ko2en 모델 하나가 함. 숫자 모델은 학습 데이터 만들 때만 사용
- 원본과 증강 중 test 정확도가 높은 쪽으로 ko2en.pt를 다시 저장

In [12]:
# en2ko 노트북의 증강 코드 그대로
@torch.no_grad()
def translate_batch(model, texts, sv, tv, batch_size=1024):
    # 여러 입력을 배치 greedy로 번역 (증강 데이터 생성용)
    outs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        src, src_len, _ = collate([(sv.encode(t), [SOS]) for t in chunk])
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        outs += [tv.decode(p) for p in pred.tolist()]
    return outs


def find_pt(name, save_dir=SAVE_DIR):
    # project01/ 에 없으면 Colab 업로드 위치(/content, 현재 폴더)도 확인
    for d in (save_dir, '.', '/content'):
        p = os.path.join(d, f'{name}.pt')
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name}.pt 없음: project01/에 merge됐는지 확인하거나 Colab 왼쪽 폴더에 업로드')


def build_augmented_pairs(data_dir=DATA_DIR, save_dir=SAVE_DIR):
    en = dict(load_pairs(os.path.join(data_dir, FILES['en'])))
    ko = dict(load_pairs(os.path.join(data_dir, FILES['ko'])))
    only_en = [k for k in en if k not in ko and split_of(k) == 'train']
    only_ko = [k for k in ko if k not in en and split_of(k) == 'train']

    gen_ko_model = load_model(find_pt('num2ko', save_dir))  # (model, sv, tv, ckpt)
    gen_en_model = load_model(find_pt('num2en', save_dir))
    assert gen_ko_model[1].lang == 'num' and gen_ko_model[2].lang == 'ko', 'num2ko.pt 언어 방향이 올바르지 않습니다.'
    assert gen_en_model[1].lang == 'num' and gen_en_model[2].lang == 'en', 'num2en.pt 언어 방향이 올바르지 않습니다.'
    t0 = time.time()
    gen_ko = translate_batch(gen_ko_model[0], only_en, *gen_ko_model[1:3])
    gen_en = translate_batch(gen_en_model[0], only_ko, *gen_en_model[1:3])
    print(f'생성 시간: {time.time() - t0:.0f}s')

    extra = ([(en[k], g, k) for k, g in zip(only_en, gen_ko)] +   # 영어 원문 + 한글 생성
             [(g, ko[k], k) for k, g in zip(only_ko, gen_en)])    # 영어 생성 + 한글 원문
    print(f'영어만 있는 train 숫자 {len(only_en):,}개 -> num2ko로 한글 생성')
    print(f'한글만 있는 train 숫자 {len(only_ko):,}개 -> num2en으로 영어 생성')
    print(f'증강 쌍 합계 {len(extra):,}개 (원본 train {len(train_loader.dataset):,}개)')
    return extra, gen_en_model, gen_ko_model



en2ko_pairs, gen_en_model, gen_ko_model = build_augmented_pairs()
extra_pairs = [(t, s, k) for s, t, k in en2ko_pairs]  # (영어, 한글) -> (한글, 영어)

short_pairs = [p for p in train_loader.dataset.pairs if len(p[2]) <= SHORT_MAX_DIGITS]
extra_pairs += short_pairs * SHORT_REPEAT  # 증강은 대부분 5자리 이상이라 짧은 숫자를 반복해서 넣음
print(f'짧은 숫자 {len(short_pairs)}개 x {SHORT_REPEAT} 추가 -> extra_train {len(extra_pairs):,}개')
for s, t, k in extra_pairs[:2] + extra_pairs[len(en2ko_pairs) - 2:len(en2ko_pairs)]:
    print(f'  [{k}] {s!r} -> {t!r}')

생성 시간: 9s
영어만 있는 train 숫자 69,866개 -> num2ko로 한글 생성
한글만 있는 train 숫자 70,159개 -> num2en으로 영어 생성
증강 쌍 합계 140,025개 (원본 train 10,162개)
짧은 숫자 806개 x 20 추가 -> extra_train 156,145개
  [3622185] '삼백육십이만 이천백팔십오' -> 'three million six hundred twenty two thousand one hundred eighty five'
  [49330605] '사천구백삼십삼만 육백오' -> 'forty nine million three hundred thirty thousand six hundred five'
  [88411795] '팔천팔백사십일만 천칠백구십오' -> 'eighty eight million four hundred eleven thousand seven hundred ninety five'
  [22878] '이만 이천팔백칠십팔' -> 'twenty two thousand eight hundred seventy eight'


In [13]:
# 원본 결과 보관
orig_model, orig_history, orig_rows = model, history, test_rows
orig_vocabs, orig_loaders, orig_max_len = (src_vocab, tgt_vocab), (train_loader, valid_loader, test_loader), MAX_LEN

train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN, extra_train=extra_pairs)
assert [p[2] for p in test_loader.dataset.pairs] == [p[2] for p in orig_loaders[2].dataset.pairs]  # test 동일
MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs + extra_pairs) + 10
print(f'train {len(orig_loaders[0].dataset):,} -> {len(train_loader.dataset):,} | '
      f'vocab src {len(src_vocab)}, tgt {len(tgt_vocab)} | 최대 길이 {MAX_LEN}')

set_seed(SEED)
aug_model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
aug_history = train(aug_model, train_loader, valid_loader, EPOCHS_AUG, LR)

extra_train: 156,145개 추가 (valid/test와 겹쳐서 제외 0개)
train 10,162 -> 166,307 | vocab src 20, tgt 36 | 최대 길이 26
epoch  1 | train loss 0.3533 | valid loss 0.0348 | valid acc 92.31% | 101s
epoch  2 | train loss 0.0454 | valid loss 0.0159 | valid acc 96.31% | 100s
epoch  3 | train loss 0.0214 | valid loss 0.0017 | valid acc 99.84% | 98s
epoch  4 | train loss 0.0085 | valid loss 0.0008 | valid acc 100.00% | 99s
epoch  5 | train loss 0.0056 | valid loss 0.0005 | valid acc 99.92% | 99s
epoch  6 | train loss 0.0040 | valid loss 0.0002 | valid acc 100.00% | 98s
epoch  7 | train loss 0.0051 | valid loss 0.0002 | valid acc 99.92% | 100s
epoch  8 | train loss 0.0036 | valid loss 0.0004 | valid acc 99.92% | 100s
epoch  9 | train loss 0.0029 | valid loss 0.0003 | valid acc 99.92% | 99s
epoch 10 | train loss 0.0020 | valid loss 0.0008 | valid acc 99.92% | 100s
best valid acc: 100.00%


In [14]:
aug_rows = predict_all(aug_model, test_loader, tgt_vocab)
print(f'test exact match (greedy): 원본 {exact_match(orig_rows)*100:.2f}% -> 증강 {exact_match(aug_rows)*100:.2f}%\n')
acc_by_length(aug_rows)
print()
show_examples(aug_rows)

sample = test_loader.dataset.pairs[:200]
b = sum(beam_decode(aug_model, s, src_vocab, tgt_vocab, BEAM_SIZE) == t for s, t, _ in sample)
print(f'증강 모델 beam({BEAM_SIZE}) test 200개: {b/len(sample)*100:.1f}%')

test exact match (greedy): 원본 99.54% -> 증강 99.92%

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 |  99.89%
     5 |    277 | 100.00%
     6 |     24 | 100.00%
     7 |      4 | 100.00%
     8 |      1 | 100.00%

--- 맞은 예시 ---
입력: 천사
정답: one thousand four
예측: one thousand four

입력: 천십
정답: one thousand ten
예측: one thousand ten

입력: 천십팔
정답: one thousand eighteen
예측: one thousand eighteen

입력: 천십구
정답: one thousand nineteen
예측: one thousand nineteen

입력: 만 이백육
정답: ten thousand two hundred six
예측: ten thousand two hundred six

--- 틀린 예시 ---
입력: 천구백팔십팔
정답: one thousand nine hundred eighty eight
예측: one hundred thousand nine hundred eighty eight

증강 모델 beam(5) test 200개: 98.5%


In [15]:
# 참고: 긴 숫자 (test는 8자리까지만 있어서 따로 확인)
# 영어 파일에만 있는 test 숫자. 정답은 원본 영어, 입력 한글은 num2ko.pt로 생성
en_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['en'])))
ko_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['ko'])))
by_digit = defaultdict(list)
for k in en_all:
    if k not in ko_all and split_of(k) == 'test' and len(k) >= 5:
        by_digit[len(k)].append(k)
rng = random.Random(SEED)
long_keys = [k for L in sorted(by_digit) for k in rng.sample(by_digit[L], min(200, len(by_digit[L])))]
long_src = translate_batch(gen_ko_model[0], long_keys, *gen_ko_model[1:3])

pred_o = translate_batch(orig_model, long_src, *orig_vocabs)
pred_a = translate_batch(aug_model, long_src, src_vocab, tgt_vocab)
print('자릿수 | 개수 |  원본   |  증강')
for L in sorted(by_digit):
    idx = [i for i, k in enumerate(long_keys) if len(k) == L]
    o = sum(pred_o[i] == en_all[long_keys[i]] for i in idx) / len(idx)
    a = sum(pred_a[i] == en_all[long_keys[i]] for i in idx) / len(idx)
    print(f'{L:6d} | {len(idx):4d} | {o*100:6.2f}% | {a*100:6.2f}%')

자릿수 | 개수 |  원본   |  증강
     5 |  200 | 100.00% | 100.00%
     6 |  200 |  95.00% | 100.00%
     7 |  200 |   9.00% |  99.50%
     8 |  200 |   0.00% |  99.50%
     9 |  200 |   0.00% |  99.00%
    10 |  200 |   0.00% |  99.00%


In [16]:
# test 정확도가 높은 쪽을 저장 (같으면 증강)
use_aug = exact_match(aug_rows) >= exact_match(orig_rows)
if use_aug:
    model, history, test_rows = aug_model, aug_history, aug_rows
else:
    model, history, test_rows = orig_model, orig_history, orig_rows
    src_vocab, tgt_vocab = orig_vocabs
    train_loader, valid_loader, test_loader = orig_loaders
    MAX_LEN = orig_max_len
print('저장할 모델:', '증강' if use_aug else '원본', f'(test {exact_match(test_rows)*100:.2f}%)')
save_model(SAVE_PATH, model, CONFIG, src_vocab, tgt_vocab,
           metrics={'test_exact_match': exact_match(test_rows), 'history': history, 'augmented': use_aug,
                    'orig_test': exact_match(orig_rows), 'aug_test': exact_match(aug_rows)})

저장할 모델: 증강 (test 99.92%)
saved: ../ko2en.pt (1.8 MB)


## 직접 구현: 입력 정규화 함수 (이재노)

모델은 데이터 파일 표기로만 학습해서, 슬라이드 표기로 넣으면 틀릴 수 있음. 그래서 입력을 데이터 표기로 바꾼 뒤 모델에 넣음.

- ko: 일만이천삼백사십오 -> 만 이천삼백사십오
- en: twelve thousands three hundreds forty five -> twelve thousand three hundred forty five
- num: 12,345 -> 12345

데이터에서 확인한 규칙
- ko: 일천/일백/일십 -> 천/백/십, 일만 -> 만, 억은 일억 유지, 띄어쓰기는 억/만 뒤에만
- en: 소문자, 복수형 없음, and·하이픈·쉼표 없음

In [17]:
import re

def normalize_ko(text):
    s = re.sub(r'[\s,]', '', text)          # 공백, 쉼표 제거
    s = re.sub(r'일([천백십])', r'\1', s)     # 일천 -> 천
    s = re.sub(r'(^|억)일만', r'\1만', s)     # 일만 -> 만 (십일만은 그대로)
    s = re.sub(r'^억', '일억', s)             # 억 -> 일억
    s = re.sub(r'([억만])(?=.)', r'\1 ', s)   # 억, 만 뒤 띄어쓰기
    return s

def normalize_en(text):
    s = re.sub(r'[-,]', ' ', text.lower())
    s = re.sub(r'\band\b', ' ', s)
    s = re.sub(r'(thousand|hundred|million|billion)s\b', r'\1', s)
    return ' '.join(s.split())

def normalize_num(text):
    return re.sub(r'[\s,_]', '', text)

def normalize(text, lang):
    if lang == 'ko':
        return normalize_ko(text)
    if lang == 'en':
        return normalize_en(text)
    return normalize_num(text)


for text, lang in [('일만이천삼백사십오', 'ko'), ('일억일만', 'ko'), ('십일만 일천', 'ko'),
                   ('twelve thousands three hundreds forty five', 'en'),
                   ('One Hundred and Twenty-One', 'en'), ('12,345', 'num')]:
    print(f'{lang:3s} {text!r:45s} -> {normalize(text, lang)!r}')

ko  '일만이천삼백사십오'                                   -> '만 이천삼백사십오'
ko  '일억일만'                                        -> '일억 만'
ko  '십일만 일천'                                      -> '십일만 천'
en  'twelve thousands three hundreds forty five'  -> 'twelve thousand three hundred forty five'
en  'One Hundred and Twenty-One'                  -> 'one hundred twenty one'
num '12,345'                                      -> '12345'


In [18]:
# 검증: 데이터 표기는 그대로, 슬라이드 표기는 데이터 표기로 돌아오는지
def to_slide_ko(text):  # 데이터 표기 -> 슬라이드 표기 (실험용)
    s = text.replace(' ', '')
    s = re.sub(r'(?<![일이삼사오육칠팔구])([십백천])', r'일\1', s)
    s = re.sub(r'(^|억)만', r'\1일만', s)
    return s

def to_slide_en(text):
    return re.sub(r'(thousand|hundred|million|billion)', r'\1s', text)

ko_all = [t for _, t in load_pairs(os.path.join(DATA_DIR, FILES['ko']))]
en_all = [t for _, t in load_pairs(os.path.join(DATA_DIR, FILES['en']))]
print('ko 그대로   :', sum(normalize(t, 'ko') == t for t in ko_all), '/', len(ko_all))
print('en 그대로   :', sum(normalize(t, 'en') == t for t in en_all), '/', len(en_all))
print('ko 슬라이드 :', sum(normalize(to_slide_ko(t), 'ko') == t for t in ko_all), '/', len(ko_all))
print('en 슬라이드 :', sum(normalize(to_slide_en(t), 'en') == t for t in en_all), '/', len(en_all))

ko 그대로   : 100000 / 100000
en 그대로   : 100000 / 100000
ko 슬라이드 : 100000 / 100000
en 슬라이드 : 100000 / 100000


### 정규화 전/후 비교
test 입력을 슬라이드 표기로 바꿔서 넣었을 때 정확도

In [19]:
def exact_match_inputs(inputs, golds):
    correct = 0
    for i in range(0, len(inputs), BATCH_SIZE):
        batch = [(src_vocab.encode(s) or [UNK], [SOS, EOS]) for s in inputs[i:i + BATCH_SIZE]]
        src, src_len, _ = collate(batch)
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p, gold in zip(pred.tolist(), golds[i:i + BATCH_SIZE]):
            correct += tgt_vocab.decode(p) == gold
    return correct / len(inputs)


test_pairs = test_loader.dataset.pairs
golds = [t for _, t, _ in test_pairs]
original = [s for s, _, _ in test_pairs]
slide = [to_slide_ko(s) for s in original]

print(f'데이터 표기          : {exact_match_inputs(original, golds)*100:.2f}%')
print(f'슬라이드 표기        : {exact_match_inputs(slide, golds)*100:.2f}%')
print(f'슬라이드 표기 + 정규화: {exact_match_inputs([normalize(s, "ko") for s in slide], golds)*100:.2f}%')

for s in ['일만이천삼백사십오', '일천일백일십일', '이만일천']:
    print(f'\n{s}')
    print('  전:', greedy_decode(model, s, src_vocab, tgt_vocab))
    print('  후:', greedy_decode(model, normalize(s, 'ko'), src_vocab, tgt_vocab))

데이터 표기          : 99.92%
슬라이드 표기        : 95.31%
슬라이드 표기 + 정규화: 99.92%

일만이천삼백사십오
  전: twelve thousand three hundred forty five
  후: twelve thousand three hundred forty five

일천일백일십일
  전: one thousand one hundred eleven
  후: one thousand one hundred eleven

이만일천
  전: twenty
  후: twenty one thousand


## 불러와서 테스트

ko2en(text)로 번역 (beam=5를 넣으면 beam search)

In [20]:
m, sv, tv, ckpt = load_model(SAVE_PATH)
print(f"loaded {ckpt['src_lang']} -> {ckpt['tgt_lang']} | test acc {ckpt['metrics']['test_exact_match']*100:.2f}%\n")
for s, t, _ in test_loader.dataset.pairs[:5]:
    print(f'입력 {s!r}\n  greedy: {greedy_decode(m, s, sv, tv)}\n  beam  : {beam_decode(m, s, sv, tv)}\n  정답  : {t}')


def ko2en(text, beam=None):
    text = normalize(text, 'ko')
    if beam:
        return beam_decode(m, text, sv, tv, beam=beam)
    return greedy_decode(m, text, sv, tv)


print()
for s in ['일만이천삼백사십오', '만 이천삼백사십오', '구천구백구십구', '일천일십', '사만 오백']:
    print(f'{s!r:24s} greedy: {ko2en(s)}')
    print(f'{"":24s} beam  : {ko2en(s, beam=5)}')

loaded ko -> en | test acc 99.92%

입력 '천사'
  greedy: one thousand four
  beam  : one thousand four
  정답  : one thousand four
입력 '천십'
  greedy: one thousand ten
  beam  : one thousand ten
  정답  : one thousand ten
입력 '천십팔'
  greedy: one thousand eighteen
  beam  : one thousand eighteen
  정답  : one thousand eighteen
입력 '천십구'
  greedy: one thousand nineteen
  beam  : one thousand nineteen
  정답  : one thousand nineteen
입력 '만 이백육'
  greedy: ten thousand two hundred six
  beam  : ten thousand two hundred six
  정답  : ten thousand two hundred six

'일만이천삼백사십오'              greedy: twelve thousand three hundred forty five
                         beam  : twelve thousand three hundred forty five
'만 이천삼백사십오'              greedy: twelve thousand three hundred forty five
                         beam  : twelve thousand three hundred forty five
'구천구백구십구'                greedy: nine thousand nine hundred ninety nine
                         beam  : nine thousand nine hundred ninety nine
'일천일십'         